In [1]:
import polars as pl
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from catboost import CatBoostClassifier, Pool


# =============================================================
# 1) AYARLAR
# =============================================================
# Bu bölüm, projenin tüm sabitlerini ve dosya yollarını merkezi olarak tanımlar.
# Böylece kodun farklı yerlerinde "sihirli sayılar" (magic numbers) kullanmaktan kaçınır,
# bakımı ve güncellenmesi kolaylaşır.

INPUT_PATH = "datathonFINAL.parquet"  # Ham veri setinin yolu.

# Çıktı dosyalarının isimleri. Bu isimlendirme standardı, dosyaların ne içerdiğini
# kod çalıştırılmadan bile belli eder ve dosya yönetimini kolaylaştırır.
OUTPUT_SCORED = "datathon_simple_scored.parquet" # Skorlanmış veri seti.
OUTPUT_SUMMARY = "simple_manipulation_summary.csv" # Şüpheli aktivitelerin özet tablosu.
OUTPUT_FI = "simple_feature_importance.csv" # Modelin özellik önemleri.
OUTPUT_MODEL = "catboost_simple_model.cbm" # Eğitilmiş modelin kendisi.

SEED = 42 # Rastgelelik içeren tüm işlemler (veri bölme, model eğitimi) için sabit bir başlangıç noktası.
          # Bu, deneyin tekrarlanabilirliğini (reproducibility) garanti altına almak için kritik öneme sahiptir.


# =============================================================
# 2) YARDIMCI FONKSİYONLAR
# =============================================================
# Kodun ana akışını temiz ve okunabilir kılmak için, belirli görevleri yerine getiren
# bağımsız fonksiyonlar tanımlanmıştır. Bu, "tek sorumluluk prensibi"ni (single responsibility principle) uygular.

def group_split(pdf: pd.DataFrame, test_size: float = 0.20, seed: int = 42):
    """
    Veriyi, aynı 'original_text'e sahip satırlar aynı bölmede kalacak şekilde
    eğitim ve test olarak böler.
    
    Neden böyle yaptık?
    - Bu bir metin manipülasyonu tespit problemidir. Aynı metnin farklı platformlardaki
      paylaşımları birbirleriyle yüksek korelasyonludur. Eğer bir metnin bir kopyası eğitimde,
      başka bir kopyası testte olursa, model metnin kendisini ezberleyerek hile yapabilir.
      Buna "veri sızıntısı" (data leakage) denir.
    - GroupShuffleSplit kullanarak, bölme işlemini `original_text` sütununa göre gruplayarak
      bu sızıntıyı engelliyoruz.
    """
    splitter = GroupShuffleSplit(
        n_splits=1, 
        test_size=test_size, 
        random_state=seed
    )

    idx_train, idx_test = next(
        splitter.split(pdf, groups=pdf["original_text"]) # Bölme kriteri olarak 'original_text' gruplarını kullan.
    )

    train = pdf.iloc[idx_train].copy().reset_index(drop=True)
    test = pdf.iloc[idx_test].copy().reset_index(drop=True)

    # Sızıntı olmadığına dair bir kontrol (assertion) ekliyoruz.
    # Bu, hata ayıklama sırasında hayat kurtarıcı bir kontroldür.
    overlap = set(train["original_text"]) & set(test["original_text"])
    assert len(overlap) == 0, f"Leakage var: {len(overlap)} ortak metin."

    return train, test


def build_text_reuse(df_pl: pl.DataFrame) -> pl.DataFrame:
    """
    Her benzersiz 'original_text' için, bu metnin veri setinde kaç kez geçtiğini
    ve kaç farklı platformda/dilde paylaşıldığını hesaplar.
    
    Neden böyle yaptık?
    - Manipülatif kampanyalar genellikle aynı içeriği birçok hesap ve platformda
      tekrar tekrar paylaşır (copy-paste, spam). Bu metrikler, bir metnin
      "kopyalanma/tekrar kullanılma" potansiyelini ölçer.
    - Bu, daha sonra 'teacher score' (öğretmen skoru) üretmek için temel sinyallerden
      biri olacak. Bu fonksiyonu ayrı yazmamızın sebebi, bu işlemi hem eğitim hem de test
      verisi için kolayca tekrar kullanabilmek.
    """
    return (
        df_pl
        .group_by("original_text") # Benzersiz her metin için grupla
        .agg([
            pl.len().alias("same_text_count"),         # Bu metin toplamda kaç kez geçiyor?
            pl.n_unique("platform").alias("text_platform_count"), # Kaç farklı platformda?
            pl.n_unique("language").alias("text_language_count"), # Kaç farklı dilde?
        ])
    )

def compute_raw_score(df_pl: pl.DataFrame) -> pl.DataFrame:
    """
    Sezgisel (heuristic) yöntemlerle bir 'teacher score' (öğretmen skoru) hesaplar.
    Bu skor, gerçek etiketlerin olmadığı durumda, modele öğrenmesi için
    'pseudo-label' (sözde etiket) üretmekte kullanılır. Zayıf denetimli (weakly supervised) bir yaklaşımdır.
    
    Neden böyle yaptık?
    - Elimizde etiketlenmiş veri yok. Denetimsiz (unsupervised) veya zayıf denetimli bir
      yaklaşıma ihtiyacımız var.
    - URL, hashtag, mention ve metin tekrarı gibi bilgiler, manipülatif davranış için
      güçlü ön bilgilerdir (prior knowledge). Bu bilgileri bir formülle birleştirerek
      bir "ham skor" oluşturuyoruz.
    - Ağırlıklar (0.45, 0.30, 0.15, 0.10) tamamen sezgisel olarak belirlenmiştir.
      URL ve hashtag gibi öğelerin spam için daha belirleyici olduğu varsayılmıştır.
    - MAX sabitleri, aykırı değerlerin (outliers) etkisini sınırlamak ve özellikleri
      normalize etmek için kullanılır. Örneğin URL sayısı 10'dan fazla olsa bile skora katkısı sabitlenir.
    """
    MAX_URL = 10.0       # URL sayısı için eşik değer
    MAX_HASHTAG = 20.0   # Hashtag sayısı için eşik değer
    MAX_MENTION = 20.0   # Mention sayısı için eşik değer
    MAX_REUSE_LOG = 10.0 # Metin tekrar sayısının logaritması için eşik değer

    return (
        df_pl
        .with_columns([
            # Her bir sinyali kendi maksimumuna bölüp 0-1 arasına sıkıştırarak normalize ediyoruz.
            # Clip ile 0 ve 1 arasında olmasını garantiliyoruz.
            (pl.col("url_count").fill_null(0) / MAX_URL)
            .clip(0.0, 1.0)
            .alias("_url_score"),

            (pl.col("hashtag_count").fill_null(0) / MAX_HASHTAG)
            .clip(0.0, 1.0)
            .alias("_hashtag_score"),

            (pl.col("mention_count").fill_null(0) / MAX_MENTION)
            .clip(0.0, 1.0)
            .alias("_mention_score"),

            # Aynı metnin çok fazla kez tekrar etmesi durumunda etkisini logaritma ile yumuşatıyoruz.
            # log1p, log(1+x) demektir ve x=0 için sorun çıkarmaz.
            (pl.col("same_text_count").fill_null(1).log1p() / MAX_REUSE_LOG)
            .clip(0.0, 1.0)
            .alias("_reuse_score"),
        ])
        .with_columns([
            # Ağırlıklandırılmış toplam ile nihai raw_score'u hesapla
            (
                0.45 * pl.col("_url_score") +
                0.30 * pl.col("_hashtag_score") +
                0.15 * pl.col("_mention_score") +
                0.10 * pl.col("_reuse_score")
            ).alias("raw_score")
        ])
        .drop([ # Geçici skor sütunlarını temizle, sadece raw_score kalsın.
            "_url_score",
            "_hashtag_score",
            "_mention_score",
            "_reuse_score",
        ])
    )


def minmax_normalize(
    df_pl: pl.DataFrame,
    train_min: float,
    train_max: float
) -> pl.DataFrame:
    """
    'raw_score' sütununu, sadece eğitim verisinden hesaplanan min ve max değerlerini
    kullanarak 0-1 aralığına normalize eder ve 'manipulation_score' olarak adlandırır.
    
    Neden böyle yaptık?
    - Model eğitimi sırasında sadece eğitim verisine bakılmalı, test verisi tamamen
      görülmemiş olmalıdır. Eğer min ve max değerlerini tüm veri seti üzerinden
      hesaplasaydık, bu test verisinden bilgi sızdırılmasına (data leakage) yol açardı.
    - Parametre olarak `train_min` ve `train_max` alır, böylece test verisi veya
      yeni gelecek veriler, eğitim verisinin istatistikleriyle normalize edilir.
      Bu, makine öğrenmesinde en iyi pratiktir (best practice).
    - `train_max == train_min` durumuna karşı bir güvenlik önlemi eklenmiştir (ZeroDivisionError engellemesi).
    """
    return df_pl.with_columns([
        pl.when(pl.lit(train_max > train_min))
        .then((pl.col("raw_score") - train_min) / (train_max - train_min))
        .otherwise(0.0)
        .clip(0.0, 1.0) # Sayısal hatalara karşı 0-1 aralığına zorla.
        .alias("manipulation_score")
    ])


def compute_safe_thresholds(
    scores: pl.Series,
    min_gap: float = 1e-5
) -> tuple:
    """
    Pseudo-label oluşturmak için güvenli bir alt (organik) ve üst (manipülatif) eşik değeri bulur.
    Amacı, modelin öğrenebileceği kadar net bir şekilde ayrışmış iki sınıf yaratmaktır.

    Neden böyle yaptık?
    - Zayıf denetimli öğrenmede, pseudo-label'ler net olmalıdır. Skor dağılımının uç
      noktalarındaki veriler "yüksek güvenle" etiketlenirken, ortada kalan belirsiz
      veriler etiketlenmez. Bu, modelin gürültülü etiketlerle eğitilmesini engeller.
    - Adım adım farklı yüzdelik dilimleri dener. Örneğin, önce en alt %10 ile en üst %90'ı
      karşılaştırır. Eğer bu iki nokta arasındaki fark yeterince büyükse (min_gap),
      arada belirgin bir ayrım var demektir ve bu eşikler kullanılır.
    - Eğer fark çok azsa, veri seti neredeyse tek bir skordan oluşuyordur.
      Bu durumda bir 'fallback' mekanizması devreye girer.
    """
    steps = [
        (0.10, 0.90), (0.15, 0.85), (0.20, 0.80),
        (0.25, 0.75), (0.30, 0.70), (0.35, 0.65), (0.40, 0.60),
    ]

    for low_pct, high_pct in steps:
        low_q = scores.quantile(low_pct)
        high_q = scores.quantile(high_pct)

        if high_q is not None and low_q is not None:
            if (high_q - low_q) >= min_gap:
                print(
                    f"Threshold seçildi: "
                    f"low_pct={low_pct}, high_pct={high_pct}, "
                    f"low={low_q:.6f}, high={high_q:.6f}"
                )
                return low_q, high_q

    # Fallback 1: Eğer skorlar yeterince ayrışmıyorsa, sadece sıfırdan büyük olanları 'manipülatif' kabul et.
    nonzero = scores.filter(scores > 0)
    if len(nonzero) > 100:
        high_q = nonzero.quantile(0.50) # Sıfırdan büyüklerin medyanı üst eşik olsun.
        if high_q is not None and high_q > min_gap:
            print(f"Fallback threshold: low=0.0, high={high_q:.6f}")
            return 0.0, high_q

    # Fallback 2: Hiçbir şey işe yaramazsa, imkansız eşikler verip tüm veriyi etiketsiz bırak.
    print("UYARI: threshold fallback aktif. score > 0 manipülatif kabul edildi.")
    return -1e-7, 1e-7


def assign_pseudo_label(
    df_pl: pl.DataFrame,
    low_q: float,
    high_q: float
) -> pl.DataFrame:
    """
    Hesaplanan eşiklere göre satırlara pseudo-label atar.
    `manipulation_score` değeri `high_q` üstünde olanlar manipülatif (1),
    `low_q` altında olanlar organik (0) olarak işaretlenir.
    Bu iki eşik arasında kalan "belirsiz" bölge ise `None` (NULL) olarak bırakılır.
    
    Neden böyle yaptık?
    - Pseudo-labeling yaklaşımının temel felsefesi budur: modeli sadece en emin olduğumuz
      örneklerle eğitmek. Belirsiz bölgedeki örnekler eğitime dahil edilmez.
    - Bu, modelin karar sınırlarını daha net çizmesine yardımcı olur.
    """
    return df_pl.with_columns([
        pl.when(pl.col("manipulation_score") >= high_q)
        .then(pl.lit(1))
        .when(pl.col("manipulation_score") <= low_q)
        .then(pl.lit(0))
        .otherwise(None) # Belirsiz bölgeyi boş bırak.
        .cast(pl.Int8)
        .alias("pseudo_label")
    ])


def cast_features(
    pdf: pd.DataFrame,
    cat_features: list,
    num_features: list
) -> pd.DataFrame:
    """
    CatBoost modeline beslemeden önce özelliklerin veri tiplerini standartlaştırır.
    Kategorik özellikleri string'e, nümerik özellikleri float'a çevirir.
    
    Neden böyle yaptık?
    - CatBoost, kategorik özellikleri string olarak alabilir, bu büyük bir avantajdır.
      Ancak veri setinde null değerler varsa sorun çıkarabilir. Bu fonksiyon,
      null'ları kategorikler için 'unknown' stringiyle, nümerikler için 0 ile doldurur.
    - Bu temizleme işlemini bir fonksiyona alarak, hem eğitim hem de test verisi için
      aynı dönüşümlerin yapılmasını sağlar ve kod tekrarını önleriz.
    """
    pdf = pdf.copy()

    for col in cat_features:
        # NaN -> "unknown", sonra tipini string yap. CatBoost'un sevdiği format.
        pdf[col] = pdf[col].fillna("unknown").astype(str)

    for col in num_features:
        # Sayısal olmayan değerleri NaN yap, sonra 0 ile doldur.
        pdf[col] = pd.to_numeric(pdf[col], errors="coerce").fillna(0)

    return pdf


def check_labels(y: pd.Series, label: str) -> bool:
    """
    Bir hedef değişkenin (y) eğitim için uygun olup olmadığını kontrol eder.
    En az iki sınıf (0 ve 1) içermesi gerekir.
    
    Neden böyle yaptık?
    - Pseudo-label oluşturma süreci başarısız olursa veya aşırı dengesizlik olursa,
      eğitim verisinde tek bir sınıf kalabilir. Bu durumda model eğitimi mantıksız
      olacağı için bir kontrol mekanizması şarttır.
    """
    counts = y.value_counts()

    print(f"\n[{label}] Label dağılımı:")
    print(counts)

    if len(counts) < 2:
        print(f"[{label}] Tek sınıf var, eğitim yapılamaz.")
        return False

    return True


def train_catboost(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_eval: pd.DataFrame,
    y_eval: pd.Series,
    cat_features: list
):
    """
    CatBoost modelini eğitir. Eğitim ve doğrulama (eval) setlerini CatBoost'un kendi
    `Pool` veri yapısına çevirir, bu da kategorik özelliklerin doğru işlenmesini sağlar.
    
    Neden böyle yaptık?
    - CatBoost, kategorik özellikleri otomatik işleyebilen, genelde çok iyi performans
      veren ve aşırı öğrenmeye karşı dayanıklı bir gradient boosting modelidir.
      Hiperparametre ayarı yapmaya gerek kalmadan bir "baseline" model için idealdir.
    - `Pool` kullanmak, `cat_features` listesini modele tanıtmanın en temiz ve hatasız yoludur.
    - `early_stopping_rounds` ile model, doğrulama setindeki AUC skoru 60 tur boyunca
      iyileşmezse eğitimi durdurur, böylece optimum noktada kalır ve overfit (aşırı öğrenme) olmaz.
    - `auto_class_weights="Balanced"` ile olası sınıf dengesizliğini otomatik olarak yönetir.
    """
    pool_train = Pool(X_train, y_train, cat_features=cat_features)
    pool_eval = Pool(X_eval, y_eval, cat_features=cat_features)

    model = CatBoostClassifier(
        iterations=800,              # Maksimum ağaç sayısı.
        learning_rate=0.05,          # Öğrenme hızı. Düşük LR, daha fazla iterasyon (ağaç) demektir.
        depth=6,                     # Ağaç derinliği. Karmaşıklığı sınırlar.
        loss_function="Logloss",     # Binary sınıflandırma için log loss.
        eval_metric="AUC",           # Başarı metriği olarak AUC kullan.
        random_seed=SEED,            # Tekrarlanabilirlik.
        auto_class_weights="Balanced", # Sınıf ağırlıklarını otomatik dengele.
        early_stopping_rounds=60,    # Erken durdurma için sabır değeri.
        verbose=100,                 # Her 100 iterasyonda bir eğitim durumunu yazdır.
    )

    print("\nModel eğitiliyor...")
    model.fit(
        pool_train,
        eval_set=pool_eval,
        use_best_model=True # Eğitim bittiğinde, en iyi performansı veren iterasyondaki modeli kullan.
    )

    return model, pool_eval


def evaluate_model(
    model: CatBoostClassifier,
    pool_eval: Pool,
    y_eval: pd.Series
):
    """
    Eğitilmiş modeli değerlendirme setinde test eder.
    
    Neden böyle yaptık?
    - Sadece "model eğitildi" demek yeterli değildir. Modelin gerçekten öğrenip öğrenmediğini
      anlamak için detaylı metrikler görmemiz gerekir.
    - Sınıflandırma raporu (F1-score, Precision, Recall), AUC ve karmaşıklık matrisi,
      modelin performansını farklı açılardan değerlendirmemizi sağlar.
    - `y_proba`'yı döndürerek ana akışta tekrar kullanılmasını sağlar.
    """
    y_proba = model.predict_proba(pool_eval)[:, 1] # Sınıf 1 (manipülatif) olma olasılığını al.
    y_pred = (y_proba >= 0.50).astype(int)         # %50 eşiği ile tahmin yap.

    print("\n--- Değerlendirme ---")
    print(classification_report(y_eval, y_pred))
    print("ROC AUC:", roc_auc_score(y_eval, y_proba))
    print("Confusion Matrix:")
    print(confusion_matrix(y_eval, y_pred))

    return y_proba


def add_reason(df_pl: pl.DataFrame) -> pl.DataFrame:
    """
    Nihai tahminin arkasındaki gerekçeyi okunabilir bir formatta ekler.
    Bu, modelin kararlarını yorumlanabilir kılmak için sonradan eklenmiş
    sezgisel kurallardır.
    
    Neden böyle yaptık?
    - Makine öğrenmesi modelleri genellikle "kara kutu"dur. Son kullanıcıya
      sadece bir skor veya etiket göstermek yetersizdir. "Neden?" sorusunun
      cevabını verebilmek (Explainable AI), sonucun güvenilirliğini artırır.
    - Bu kurallar `compute_raw_score`'daki mantıkla paralellik gösterir ve
      en belirgin sinyallere dayanır.
    """
    return df_pl.with_columns([
        pl.when(pl.col("url_count").fill_null(0) >= 3)
        .then(pl.lit("Cok sayida link iceriyor"))
        .when(pl.col("hashtag_count").fill_null(0) >= 8)
        .then(pl.lit("Cok sayida hashtag iceriyor"))
        .when(pl.col("mention_count").fill_null(0) >= 8)
        .then(pl.lit("Cok sayida mention iceriyor"))
        .when(pl.col("same_text_count").fill_null(1) >= 10)
        .then(pl.lit("Ayni metin cok kez kullanilmis"))
        .otherwise(pl.lit("Normal veya belirsiz davranis sinyalleri"))
        .alias("reason")
    ])


# =============================================================
# 3) VERİYİ OKU VE TEMİZLE
# =============================================================
# Bu bölüm, ham veriyi okur ve bir dizi özellik mühendisliği (feature engineering)
# ve temizleme adımından geçirir.

print("Veri okunuyor...")

df = pl.read_parquet(INPUT_PATH) # Polars ile hızlı okuma. Parquet formatı performanslıdır.

df = (
    df
    .filter( # Boş metinleri filtrele. Metni olmayan bir gönderiyi incelemek anlamsızdır.
        pl.col("original_text").is_not_null() &
        (pl.col("original_text") != "")
    )
    .with_columns([ # Yeni özellikler türet.
        pl.col("date")
        .str.to_datetime(format="%Y-%m-%dT%H:%M:%S%.3fZ", strict=False) # Standard datetime formatına çevir.
        .alias("dt"),

        pl.col("url")
        .str.extract(r"(?:https?://)?(?:www\.)?([^/]+)", 1) # URL'den domain (platform) adını regex ile çıkar.
        .str.to_lowercase()
        .fill_null("unknown")
        .alias("platform"),

        pl.col("original_text")
        .str.len_chars()
        .alias("text_len"), # Metin uzunluğu. Spam/kısa mesajlar veya anormal uzunluklar bilgi verir.

        pl.col("original_text")
        .str.count_matches(r"#\w+")
        .alias("hashtag_count"), # Hashtag sayısı. Spam için güçlü bir sinyaldir.

        pl.col("original_text")
        .str.count_matches(r"https?://|www\.")
        .alias("url_count"), # URL sayısı. Spam için bir diğer güçlü sinyal.

        pl.col("original_text")
        .str.count_matches(r"@[\w_]+")
        .alias("mention_count"), # Mention sayısı.

        pl.col("sentiment")
        .cast(pl.Float64, strict=False) # Duygu analizi skorunu sayısala çevir.
        .fill_null(0.0)
        .alias("sentiment"),

        pl.col("language").fill_null("unknown").alias("language"), # Null dil değerlerini doldur.
        pl.col("primary_theme").fill_null("unknown").alias("primary_theme"), # Null tema değerlerini doldur.
        pl.col("main_emotion").fill_null("unknown").alias("main_emotion"), # Null duygu değerlerini doldur.
    ])
)

print(f"Toplam satır: {len(df):,}")


# =============================================================
# 4) TRAIN / TEST SPLIT
# =============================================================
# Veri sızıntısını önlemek için group_split fonksiyonumuzu kullanıyoruz.

# Polars DataFrame'ini geçici olarak Pandas'a çeviriyoruz çünkü GroupShuffleSplit Pandas ile çalışır.
pdf = df.to_pandas()

train_pdf, test_pdf = group_split(pdf, test_size=0.20, seed=SEED)

# Böldükten sonra verimizi tekrar ana formatımız olan Polars'a geri çeviriyoruz.
# Bu hafif bir maliyet getirse de, Polars'ın sorgu optimizasyonu ve hızından
# yararlanmaya devam etmek için değer.
df_train = pl.from_pandas(train_pdf)
df_test = pl.from_pandas(test_pdf)

print(f"Train: {len(df_train):,}")
print(f"Test : {len(df_test):,}")


# =============================================================
# 5) TEACHER SCORE VE PSEUDO LABEL
# =============================================================
# Zayıf denetimli öğrenme akışının kalbidir.
# Önce eğitim verisi için bir "öğretmen skoru" hesaplanır, normalize edilir,
# eşik değerler bulunur ve pseudo-label'ler atanır.

print("\nText reuse feature hazırlanıyor...")
text_reuse = build_text_reuse(df_train) # Eğitim verisindeki metin tekrar istatistiklerini hesapla.

df_train_scored = (
    df_train
    .join(text_reuse, on="original_text", how="left") # Bu istatistikleri ana veriye bağla.
    .pipe(compute_raw_score) # Ham skoru hesapla.
)

# Test verisini de normalize edip etiketlerken kullanmak üzere, eğitimin min ve max değerlerini sakla.
train_min = df_train_scored["raw_score"].min()
train_max = df_train_scored["raw_score"].max()

df_train_scored = minmax_normalize(df_train_scored, train_min, train_max) # 0-1 arası manipülasyon skorunu hesapla.

print("\nTrain manipulation_score dağılımı:")
print(df_train_scored["manipulation_score"].describe())

# Pseudo-label oluşturmak için güvenli eşikleri hesapla.
low_q, high_q = compute_safe_thresholds(df_train_scored["manipulation_score"])

# Eğitim verisine pseudo-label ata. Burada train_min/train_max kullanıldığına dikkat!
df_train_scored = assign_pseudo_label(df_train_scored, low_q, high_q)

# Test verisi için de AYNI adımları tekrarla, ancak normalize ederken EĞİTİM VERİSİNİN min/max değerlerini kullan.
# Bu, veri sızıntısını önleyen kritik bir adımdır.
df_test_scored = (
    df_test
    .join(text_reuse, on="original_text", how="left")
    .pipe(compute_raw_score)
    .pipe(minmax_normalize, train_min, train_max)
    .pipe(assign_pseudo_label, low_q, high_q)
)


# =============================================================
# 6) MODEL FEATURE SETİ
# =============================================================
# Modelin öğrenmesini istediğimiz asıl özellikleri seçiyoruz.
# DİKKAT: Öğretmen skorunun kullandığı url_count, hashtag_count gibi doğrudan sinyalleri
# model özelliklerine BİLEREK koymuyoruz.
# Amacımız, modelin sadece metin ve platform özelliklerine bakarak öğretmen skorunu
# taklit etmeyi öğrenmesidir. Bu, modelin daha genellenebilir ve daha az "hileci" olmasını sağlar.

CAT_FEATURES = [
    "language",        # Metnin dili. Bazı manipülasyon kampanyaları belirli dilleri hedefler.
    "platform",        # Gönderinin paylaşıldığı platform (Twitter, Facebook vs.).
    "primary_theme",   # Metnin ana teması. Belirli konular manipülasyona daha yatkın olabilir.
    "main_emotion",    # Metnin baskın duygusu. Öfke veya korku gibi duygular manipülasyonda sık kullanılır.
]

NUM_FEATURES = [
    "text_len",  # Metin uzunluğu. Çok kısa veya çok uzun metinler anomali gösterebilir.
    "sentiment", # Duygu analizi skoru (-1 negatif, +1 pozitif).
]

FEATURES = CAT_FEATURES + NUM_FEATURES

# Eğitim ve test verisinden sadece etiketlenmiş (belirsiz olmayan) satırları seç.
train_labeled = df_train_scored.filter(pl.col("pseudo_label").is_not_null())
test_labeled = df_test_scored.filter(pl.col("pseudo_label").is_not_null())

# Özellikleri CatBoost için hazırla (tip dönüşümleri, null doldurma vs.).
X_train = cast_features(train_labeled.select(FEATURES).to_pandas(), CAT_FEATURES, NUM_FEATURES)
y_train = train_labeled["pseudo_label"].to_pandas().astype(int)
X_eval = cast_features(test_labeled.select(FEATURES).to_pandas(), CAT_FEATURES, NUM_FEATURES)
y_eval = test_labeled["pseudo_label"].to_pandas().astype(int)

print(f"\nEtiketli train: {len(X_train):,}")
print(f"Etiketli eval : {len(X_eval):,}")


# =============================================================
# 7) MODEL EĞİT
# =============================================================
# Etiket kontrollerini geçerse modeli eğit, değerlendir ve kaydet.

model = None

if check_labels(y_train, "train") and check_labels(y_eval, "eval"):
    model, pool_eval = train_catboost(X_train, y_train, X_eval, y_eval, CAT_FEATURES)

    evaluate_model(model, pool_eval, y_eval)

    # Feature importance: Modelin hangi özelliklere ne kadar güvendiğini gösterir.
    # Bu, modelin mantıklı öğrenip öğrenmediğini anlamak için çok değerlidir.
    feature_importance = pd.DataFrame({
        "feature": FEATURES,
        "importance": model.get_feature_importance()
    }).sort_values("importance", ascending=False)

    print("\nFeature Importance:")
    print(feature_importance)
    feature_importance.to_csv(OUTPUT_FI, index=False)

    model.save_model(OUTPUT_MODEL) # Eğitilmiş modeli diske kaydet.
    print(f"\nModel kaydedildi: {OUTPUT_MODEL}")

else:
    # Eğitim başarısız olursa, model olmadan devam et.
    print("\nModel eğitilemedi. Heuristic skor kullanılacak.")
    feature_importance = pd.DataFrame({"feature": FEATURES, "importance": 0.0})
    feature_importance.to_csv(OUTPUT_FI, index=False)


# =============================================================
# 8) TÜM VERİYE SKOR UYGULA
# =============================================================
# Eğitilmiş modeli (veya fallback olarak öğretmen skorunu) tüm veri setine uygula.

print("\nTüm veri skorlanıyor...")
df_full_scored = (
    df
    .join(text_reuse, on="original_text", how="left")
    .pipe(compute_raw_score)
    .pipe(minmax_normalize, train_min, train_max) # Yine eğitim min/max'i ile normalize et.
    .pipe(assign_pseudo_label, low_q, high_q)
)

if model is not None:
    # Model varsa, tüm veri için tahmin olasılığını hesapla.
    X_full = cast_features(df_full_scored.select(FEATURES).to_pandas(), CAT_FEATURES, NUM_FEATURES)
    full_scores = model.predict_proba(Pool(X_full, cat_features=CAT_FEATURES))[:, 1]
else:
    # Model yoksa, teacher score'u nihai skor olarak kullan.
    full_scores = df_full_scored["manipulation_score"].to_numpy()

# Yeni sütunları ekle: model skoru ve organik skoru (1 - manipülasyon skoru).
df_full_scored = (
    df_full_scored
    .with_columns([
        pl.Series("model_manipulation_score", full_scores),
        pl.Series("model_organic_score", 1.0 - full_scores),
    ])
)

print("\nModel score dağılımı:")
print(df_full_scored["model_manipulation_score"].describe())


# =============================================================
# 9) FINAL PREDICTION
# =============================================================
# 'Belirsiz' bölgeyi de hesaba katarak, nihai sınıflandırmayı yapıyoruz.
# Bu adım, modelin ham skorunu yorumlanabilir bir karara dönüştürür.

# Model skorunun dağılımına bağlı dinamik eşikler belirle.
# Bu, sabit bir 0.5 eşiğinden daha esnek ve veri setine özgüdür.
model_score_low_q = df_full_scored["model_manipulation_score"].quantile(0.10)
model_score_high_q = df_full_scored["model_manipulation_score"].quantile(0.90)

print(f"\nFinal model eşikleri:")
print(f"Organic <= {model_score_low_q:.6f}")
print(f"Manipulatif >= {model_score_high_q:.6f}")

df_full_scored = (
    df_full_scored
    .with_columns([
        # Karar mantığı:
        # 'Manipulatif' olması için: model skoru yüksek eşiğin üstünde OLMALI
        #                   VE ( teacher skoru yüksek VEYA en az bir spam sinyali pozitif OLMALI).
        # Bu, hem modelin hem de sezgisel kuralların hemfikir olduğu durumları yakalar,
        # precision'ı (kesinliği) artırır ve yanlış pozitifleri azaltır.
        pl.when(
            (pl.col("model_manipulation_score") >= model_score_high_q) &
            (
                (pl.col("manipulation_score") >= high_q) |
                (pl.col("url_count").fill_null(0) >= 1) |
                (pl.col("hashtag_count").fill_null(0) >= 3) |
                (pl.col("mention_count").fill_null(0) >= 3) |
                (pl.col("same_text_count").fill_null(1) >= 5)
            )
        )
        .then(pl.lit("Manipulatif"))

        # 'Organik' olması için model skorunun düşük eşiğin altında olması yeterli.
        .when(pl.col("model_manipulation_score") <= model_score_low_q)
        .then(pl.lit("Organik"))

        # Arada kalan her şey 'Belirsiz'dir.
        .otherwise(pl.lit("Belirsiz"))
        .alias("model_prediction")
    ])
    .pipe(add_reason) # Kararın gerekçesini ekle.
)

print("\nPrediction dağılımı:")
print(df_full_scored["model_prediction"].value_counts())


# =============================================================
# 10) ÖZET TABLO
# =============================================================
# İş birimlerine sonuçları göstermek için, manipülatif olarak işaretlenmiş
# gönderilerin platform, dil ve temaya göre bir özet tablosunu oluştur.

summary = (
    df_full_scored
    .filter(pl.col("model_prediction") == "Manipulatif")
    .group_by(["language", "platform", "primary_theme"])
    .agg([
        pl.len().alias("suspicious_count"),
        pl.mean("model_manipulation_score").alias("avg_model_score"),
        pl.mean("manipulation_score").alias("avg_teacher_score"),
        pl.n_unique("original_text").alias("unique_texts"),
    ])
    .sort("suspicious_count", descending=True)
)

print("\nSummary ilk 20:")
print(summary.head(20))


# =============================================================
# 11) TOP ŞÜPHELİ ÖRNEKLER
# =============================================================
# En yüksek skorlu 20 şüpheli gönderiyi incelemek için seç.

top_suspicious = (
    df_full_scored
    .filter(pl.col("model_prediction") == "Manipulatif")
    .select([ # İlgili sütunları seç ve skora göre sırala.
        "original_text", "language", "platform", "primary_theme",
        "main_emotion", "sentiment", "text_len",
        "url_count", "hashtag_count", "mention_count", "same_text_count",
        "manipulation_score", "model_manipulation_score", "reason",
    ])
    .sort("model_manipulation_score", descending=True)
    .head(20)
)

print("\nTop şüpheli örnekler:")
print(top_suspicious)


# =============================================================
# 12) KAYDET
# =============================================================
# Tüm çıktıları diske yaz.

df_full_scored.write_parquet(OUTPUT_SCORED)
summary.write_csv(OUTPUT_SUMMARY)

print("\nKaydedildi:")
print(f"- {OUTPUT_SCORED}")
print(f"- {OUTPUT_SUMMARY}")
print(f"- {OUTPUT_FI}")

if model is not None:
    print(f"- {OUTPUT_MODEL}")

# =============================================================
# 13) TRAINING DATA EXPORT (SENİN İSTEDİĞİN FORMAT)
# =============================================================
# İstenilen formatta, sadece metin ve ikili etiket içeren bir eğitim verisi oluşturur.
# Bu format, başka modellerin (örneğin bir transformer) eğitilmesi için girdi olabilir.

training_df = (
    df_full_scored
    .filter(pl.col("model_prediction") != "Belirsiz") # Sadece emin olduklarımızı al.
    .with_columns([
        pl.when(pl.col("model_prediction") == "Manipulatif")
        .then(1)
        .otherwise(0)
        .alias("label")
    ])
    .select([
        "original_text",
        "label"
    ])
)

print("\nTraining dataset hazır:")
print(training_df.shape)

training_df.write_csv("training_data_for_model.csv")

print("\nKaydedildi:")
print("- training_data_for_model.csv")

Veri okunuyor...
Toplam satır: 5,004,812
Train: 4,003,550
Test : 1,001,262

Text reuse feature hazırlanıyor...

Train manipulation_score dağılımı:
shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ value     │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 4.00355e6 │
│ null_count ┆ 0.0       │
│ mean       ┆ 0.010811  │
│ std        ┆ 0.038662  │
│ min        ┆ 0.0       │
│ 25%        ┆ 0.0       │
│ 50%        ┆ 0.0       │
│ 75%        ┆ 0.0       │
│ max        ┆ 1.0       │
└────────────┴───────────┘
Threshold seçildi: low_pct=0.1, high_pct=0.9, low=0.000000, high=0.025000

Etiketli train: 3,733,729
Etiketli eval : 977,783

[train] Label dağılımı:
pseudo_label
0    3260268
1     473461
Name: count, dtype: int64

[eval] Label dağılımı:
pseudo_label
0    871928
1    105855
Name: count, dtype: int64

Model eğitiliyor...
0:	test: 0.7709371	best: 0.7709371 (0)	total: 1.99s	remaining: 26m 31s
100:	test: 0.8489396	best: 0.8489396 (10

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score


# =============================================================
# 1) AYARLAR
# =============================================================
# Bu bölüm, tüm dosya yolları ve hiperparametre sabitlerini merkezileştirir.
# Amacı: "Bu dosya nereden geliyor, çıktılar nereye gidiyor?" sorusunu
# kodun geri kalanını okumaya gerek kalmadan cevaplayabilmektir.

INPUT_PATH = "training_data_for_model.csv"
# Bu dosya, catboost_simple_model'ın ürettiği pseudo-label'li eğitim verisidir.
# Sadece "original_text" ve "label" (0/1) sütunlarını içerir.
# Amacı: Zayıf denetimli öğrenme ile üretilen etiketleri, daha güçlü bir
# metin modeline aktarmak için bir köprü görevi görmesidir.

OUTPUT_MODEL = "tfidf_logreg_model.joblib"
# Eğitilmiş pipeline'ı (TF-IDF + LogReg) diske kaydetmek için.
# .joblib uzantısı, sklearn pipeline'ları gibi karmaşık Python nesnelerini
# serialize etmek için en uygun formattır (pickle'dan daha hızlı ve güvenlidir).

OUTPUT_PREDICTIONS = "tfidf_validation_predictions.csv"
# Validation seti üzerindeki tahminleri, gerçek etiketlerle karşılaştırmalı
# olarak kaydetmek için. Hata analizi (error analysis) yaparken kullanılacak.

SEED = 42
# Rastgelelik içeren tüm işlemler (train/test split, model başlatma) için sabit
# başlangıç noktası. Bu olmadan, her çalıştırmada farklı sonuç alır ve
# modelin gerçek performansını değerlendiremezdik.


# =============================================================
# 2) VERİYİ OKU
# =============================================================
# Ham veriyi okuyup, modelin beklediği temel temizlik adımlarını uygular.

print("Veri okunuyor...")

df = pd.read_csv(INPUT_PATH)

# NaN içeren kritik sütunları temizle.
# 'original_text' olmadan model çalışamaz, 'label' olmadan eğitim yapılamaz.
# Bu satır, veri setindeki olası bozukluklara karşı bir güvenlik önlemidir.
df = df.dropna(subset=["original_text", "label"])

# Veri tiplerini garanti altına al.
# Metin sütunu string olmazsa TF-IDF hata verir.
# Etiket integer olmazsa LogisticRegression'ın class_weight hesabı bozulabilir.
df["original_text"] = df["original_text"].astype(str)
df["label"] = df["label"].astype(int)

print("Veri boyutu:", df.shape)

# Etiket dağılımını kontrol etmek, model eğitiminin ilk adımıdır.
# Dengesiz bir veri seti (örneğin %95 organik, %5 manipülatif) ile karşılaşırsak,
# class_weight='balanced' kullanmamız gerektiğini biliriz. Bu çıktı bize yol gösterir.
print("\nLabel dağılımı:")
print(df["label"].value_counts())


# =============================================================
# 3) TRAIN / VALIDATION SPLIT
# =============================================================
# Modeli eğitmek ve değerlendirmek için veriyi böleriz.
# Gerçek dünyada test seti ayrıdır; validation seti, eğitim sırasında
# modelin genelleme performansını ölçmek için kullanılır.

X_train, X_val, y_train, y_val = train_test_split(
    df["original_text"],  # Özellik: Ham metin (TF-IDF bunu işleyecek)
    df["label"],          # Hedef: 0 (organik) veya 1 (manipülatif)
    test_size=0.20,       # Verinin %80'i eğitim, %20'si doğrulama için.
    random_state=SEED,    # Tekrarlanabilirlik için SEED kullan.
    stratify=df["label"]  # KRİTİK: Etiket dağılımını train ve val setlerinde koru.
)
# Neden stratify kullanıyoruz?
# Eğer veri setinde %90 organik (0), %10 manipülatif (1) varsa, rastgele bölme
# sırasında validation setine hiç manipülatif örnek düşmeyebilir. Bu durumda
# model değerlendirmesi anlamsız olur. Stratify, bu dağılımın aynen korunmasını sağlar.

print("\nTrain:", len(X_train))
print("Validation:", len(X_val))


# =============================================================
# 4) TF-IDF FEATURE SET
# =============================================================
# Metni sayısal vektörlere çevirmek için TF-IDF kullanıyoruz.
# Burada iki farklı TF-IDF vektörleştiriciyi FeatureUnion ile birleştiriyoruz.
# Her ikisi de aynı metni farklı açılardan temsil eder; birleşimleri daha
# zengin ve gürbüz bir özellik uzayı oluşturur.

word_tfidf = TfidfVectorizer(
    analyzer="word",            # Metni kelimelere (token) ayırır.
    ngram_range=(1, 2),         # Tekil kelimeler (unigram) ve ikili kelime grupları (bigram).
                                # "not good" bigram'ı, "not" ve "good" unigramlarından farklı anlam taşır.
    max_features=300_000,       # En sık kullanılan 300.000 özelliği tut, gerisini at.
                                # Boyut kontrolü ve gürültü azaltma için.
    min_df=3,                   # En az 3 dokümanda geçmeyen kelimeleri yok say.
                                # "typo" veya çok nadir kullanıcı adları gibi gürültüleri filtreler.
    max_df=0.95,                # Dokümanların %95'inden fazlasında geçen kelimeleri yok say.
                                # "the", "a" gibi ayırt edici olmayan "stop word" benzeri kelimeleri filtreler.
    sublinear_tf=True,          # Ham terim frekansı yerine 1 + log(tf) kullan.
                                # Bir kelimenin 100 kez geçmesi, 1 kez geçmesinden 100 kat daha önemli
                                # olmamalı; logaritmik ölçeklendirme bu aşırılığı törpüler.
    strip_accents=None,         # Aksanları kaldırma. None: olduğu gibi bırak.
                                # Sosyal medya metninde aksanlar önemli olabilir (ör. İngilizce vs. Fransızca).
    lowercase=True              # Tüm metni küçük harfe çevir. "Manipulation" ve "manipulation" aynı kabul edilsin.
)

char_tfidf = TfidfVectorizer(
    analyzer="char_wb",         # Karakter n-gram, sadece kelime sınırları içinde (word boundary).
                                # "char" yerine "char_wb" kullanmamızın nedeni:
                                # Kelimelerin başlangıç ve bitişlerine odaklanır, ortadan anlamsız
                                # n-gram'lar üretmez. "ing" yerine " ing" (başta boşluk) gibi.
    ngram_range=(3, 5),         # 3'lü, 4'lü ve 5'li karakter grupları.
                                # Çok dilli ortamda MUCİZEVİ çalışır. Örneğin "tion", "ment", "lik"
                                # gibi ekler veya "#hashtag" içindeki "#ha" gibi karakter dizileri,
                                # dil bağımsız ipuçları taşır. Manipülatif kampanyalardaki
                                # yazım hatalarını veya kendine has kalıpları yakalar.
    max_features=300_000,       # Word TF-IDF ile aynı mantık: boyut kontrolü.
    min_df=3,                   # En az 3 dokümanda geçen karakter n-gram'ları tut.
    max_df=0.95,                # Çok sık geçen karakter dizilerini filtrele.
    sublinear_tf=True,          # Logaritmik ölçeklendirme.
    lowercase=True              # Karakterleri de küçük harfe çevir.
)

# FeatureUnion: İki vektörleştiricinin çıktısını yatay olarak birleştirir (hstack).
# Sonuç: Her bir metin için [word_tfidf özellikleri | char_tfidf özellikleri] şeklinde
# tek bir dev vektör. Model, hangi özellik grubunun daha faydalı olduğunu
# ağırlıklar üzerinden kendi öğrenir.
features = FeatureUnion([
    ("word_tfidf", word_tfidf),  # İlk bölüm: kelime tabanlı özellikler
    ("char_tfidf", char_tfidf),  # İkinci bölüm: karakter tabanlı özellikler
])


# =============================================================
# 5) MODEL
# =============================================================
# TF-IDF + Lojistik Regresyon, metin sınıflandırmada "baseline" için altın standarttır.
# Yüksek boyutlu, seyrek (sparse) verilerde inanılmaz hızlı ve etkilidir.
# Derin öğrenme modellerine geçmeden önce, ne kadar başarılı olabileceğimizin
# üst sınırını görmek için idealdir.

model = LogisticRegression(
    C=2.0,
    # Düzenlileştirme (regularization) şiddetinin tersi. C ne kadar küçükse, düzenlileştirme
    # o kadar güçlüdür (aşırı öğrenmeyi engeller). C=2.0, varsayılan C=1.0'a göre
    # biraz daha esnek bir model demektir. 300.000 özellikle çalıştığımız için
    # aşırı öğrenme riski yüksektir; C'yi çok büyük seçmedik.

    max_iter=1000,
    # Maksimum iterasyon sayısı. Varsayılan 100, yüksek boyutlu TF-IDF verisinde
    # genelde yetersiz kalır ve "STOP: TOTAL NO. of ITERATIONS REACHED LIMIT."
    # uyarısı verir. 1000, yakınsama (convergence) için yeterli bir tampondur.

    solver="saga",
    # Optimizasyon algoritması. "saga" seçilmesinin özel nedenleri:
    # 1) Büyük veri setlerinde hızlıdır (stochastic average gradient).
    # 2) SEYREK (sparse) verilerle çalışabilir. TF-IDF matrisimiz milyonlarca
    #    sütunlu ama çoğu hücresi 0 olan bir yapıdadır. "saga" bu yapıyı verimli işler.
    # 3) L1 veya Elastic Net düzenlileştirmeyi de destekler (ileride gerekirse).

    class_weight="balanced",
    # SINIF DENGESİZLİĞİ İÇİN KRİTİK PARAMETRE!
    # Eğitim verisinde organik (0) ve manipülatif (1) sınıflarının sayısı eşit değilse,
    # model çoğunluk sınıfına yönelebilir. "balanced" parametresi, azınlık sınıfındaki
    # hatalara daha fazla ceza vererek bu eğilimi otomatik olarak dengeler.
    # Ağırlıklar: n_samples / (n_classes * np.bincount(y)) formülü ile otomatik hesaplanır.

    n_jobs=-1,
    # İşlemci çekirdeklerinin hepsini kullan. Eğitim süresini önemli ölçüde kısaltır.
    # Özellikle cross-validation gibi tekrarlı işlemlerde farkı hissedilir.

    verbose=1,
    # Eğitim sürecini konsola yazdır. "Kaç iterasyon kaldı, yakınsadı mı?" gibi soruları
    # cevaplar. Sessizce beklemekten iyidir.

    random_state=SEED
    # Farklı solver'lar farklı başlangıç noktaları kullanabilir. Bunu sabitleyerek
    # aynı veriden her zaman aynı modelin çıkmasını sağlarız.
)

# Pipeline: Veri dönüşümü (features) ve modeli (model) tek bir nesnede birleştirir.
# Bu yapının avantajı: fit() dediğimizde önce TF-IDF öğrenilir, metinler vektöre çevrilir,
# sonra bu vektörlerle lojistik regresyon eğitilir.
# predict() dediğimizde ise aynı dönüşümler otomatik olarak uygulanır.
# Bu, manuel adımlardan kaynaklanacak hataları (örneğin test verisine fit_transform
# yerine transform uygulamayı unutmak) kesin olarak engeller.
pipeline = Pipeline([
    ("features", features),  # 1. Adım: Metin -> TF-IDF vektörü
    ("model", model)         # 2. Adım: TF-IDF vektörü -> Sınıf tahmini
])


# =============================================================
# 6) EĞİTİM
# =============================================================
# Pipeline'ın fit() metodu tüm sihri yapar:
# 1. X_train metinlerini alır, kelime/karakter dağarcığını (vocabulary) öğrenir.
# 2. X_train'i TF-IDF matrisine çevirir.
# 3. Bu matris ve y_train etiketleriyle Lojistik Regresyon'u eğitir.

print("\nModel eğitiliyor...")
pipeline.fit(X_train, y_train)


# =============================================================
# 7) DEĞERLENDİRME
# =============================================================
# Modelin daha önce hiç görmediği validation seti üzerindeki performansını ölçer.
# Bu, modelin gerçek dünyadaki başarısının en iyi tahminidir.

print("\nValidation tahminleri alınıyor...")

# predict_proba, her sınıfa ait olasılığı döndürür. [:, 1] ile sadece
# "manipülatif" (sınıf 1) olma olasılığını alıyoruz.
val_proba = pipeline.predict_proba(X_val)[:, 1]

# %50 eşik değeri ile olasılığı ikili (0/1) tahmine çeviriyoruz.
# Bu eşik, ileride ihtiyaca göre precision/recall dengesi için ayarlanabilir.
val_pred = (val_proba >= 0.50).astype(int)

print("\n--- Validation Report ---")
# Precision: Manipülatif dediklerimizin gerçekten ne kadarı manipülatif? (Yanlış alarm oranı)
# Recall: Gerçek manipülatiflerin ne kadarını yakaladık? (Kaçırma oranı)
# F1-score: Precision ve Recall'un harmonik ortalaması. Dengesiz veri setlerinde
#           accuracy'den daha güvenilir bir metriktir.
print(classification_report(y_val, val_pred))

# ROC AUC: Modelin farklı eşik değerlerindeki ayrım gücünü ölçer.
# 0.5 = rastgele tahmin, 1.0 = mükemmel ayrım. Sınıf dengesizliğinden etkilenmez,
# bu yüzden dengesiz veri setlerinde en güvenilir metriklerden biridir.
print("ROC AUC:", roc_auc_score(y_val, val_proba))

print("\nConfusion Matrix:")
# Satırlar: Gerçek etiket (0,1)
# Sütunlar: Tahmin edilen etiket (0,1)
# [[TN, FP],
#  [FN, TP]]
# TP (sağ alt): Doğru yakalanan manipülatifler.
# FP (sağ üst): Yanlış alarmlar (organik ama manipülatif denmiş).
print(confusion_matrix(y_val, val_pred))


# =============================================================
# 8) VALIDATION ÇIKTISI KAYDET
# =============================================================
# Tahminleri bir CSV dosyasına kaydetmemizin iki önemli nedeni var:
# 1) HATA ANALİZİ: Yanlış tahmin edilen örnekleri açıp okuyarak modelin
#    nerede hata yaptığını anlayabilir, yeni özellikler veya kurallar
#    geliştirebiliriz.
# 2) İŞ BİRİMİ SUNUMU: Teknik olmayan ekip arkadaşları, Excel'de açıp
#    örnekleri inceleyerek modele güven duyabilir.
pred_df = pd.DataFrame({
    "original_text": X_val.values,       # Ham metin
    "true_label": y_val.values,          # Gerçek etiket (0/1)
    "pred_label": val_pred,              # Modelin ikili tahmini
    "manipulation_probability": val_proba # Modelin verdiği olasılık skoru
})

pred_df.to_csv(OUTPUT_PREDICTIONS, index=False)

print(f"\nValidation tahminleri kaydedildi: {OUTPUT_PREDICTIONS}")


# =============================================================
# 9) MODELİ KAYDET
# =============================================================
# Eğitilmiş pipeline'ı diske kaydediyoruz. Bu sayede:
# 1) Modeli tekrar eğitmek zorunda kalmayız (eğitim saatler sürebilir).
# 2) Bir API'nin arkasına koyarak gerçek zamanlı tahmin yapabiliriz.
# 3) Model versiyonlaması yapabiliriz (tfidf_logreg_v1.joblib, v2.joblib...).
# joblib, sklearn'in önerdiği serileştirme (serialization) kütüphanesidir.
# pickle'a göre büyük numpy array'lerini daha verimli sıkıştırır.

joblib.dump(pipeline, OUTPUT_MODEL)

print(f"Model kaydedildi: {OUTPUT_MODEL}")

Veri okunuyor...
Veri boyutu: (765226, 2)

Label dağılımı:
label
0    500482
1    264744
Name: count, dtype: int64

Train: 612180
Validation: 153046

Model eğitiliyor...


c:\Users\memin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


convergence after 25 epochs took 61 seconds

Validation tahminleri alınıyor...

--- Validation Report ---
              precision    recall  f1-score   support

           0       0.98      0.99      0.98    100097
           1       0.98      0.97      0.97     52949

    accuracy                           0.98    153046
   macro avg       0.98      0.98      0.98    153046
weighted avg       0.98      0.98      0.98    153046

ROC AUC: 0.9976799115370094

Confusion Matrix:
[[98839  1258]
 [ 1813 51136]]

Validation tahminleri kaydedildi: tfidf_validation_predictions.csv
Model kaydedildi: tfidf_logreg_model.joblib


In [ ]:
import polars as pl
import joblib


def run_inference(
    input_path: str,
    model_path: str = "tfidf_logreg_model.joblib",
    text_col: str = "text",
    id_col: str = "test_id",
    output_path: str = "submission.csv"
):
    """
    Test datası üzerinde inference yapar.

    Bu fonksiyon, eğitilmiş TF-IDF + Logistic Regression pipeline'ını kullanarak
    yeni gelen metinler üzerinde manipülasyon olasılığını tahmin eder.
    
    Neden ayrı bir inference script'i?
    - Eğitim ve tahmin (inference) süreçleri farklı ortamlarda çalışabilir.
      Eğitim GPU'lu bir makinede, inference hafif bir API sunucusunda yapılabilir.
    - Bu script, model dosyası (.joblib) dışında hiçbir eğitim bağımlılığı taşımaz.
      Sadece joblib ve polars yeterlidir.
    - Production ortamında, eğitim kodunun tamamını deploy etmek gereksizdir.
      Sadece bu hafif inference fonksiyonu ve model dosyası yeterlidir.

    Senaryolar:
    1) id varsa → id + probability
       (Yarışma formatı: gönderi ID'si ve tahmin skoru)
    2) id yoksa → text + probability
       (Analiz formatı: hangi metin hangi skoru almış, incelemek için)

    Output:
    probability = manipulatif olma olasılığı (0-1)
    """

    print("Veri okunuyor...")
    df = pl.read_csv(input_path)
    # Polars kullanma sebebimiz:
    # - Büyük test dosyalarında (milyonlarca satır) Pandas'tan çok daha hızlıdır.
    # - Bellek kullanımı daha verimlidir.
    # - API entegrasyonlarında düşük gecikme süresi kritik olduğu için performans önemlidir.

    print(f"Satır sayısı: {len(df):,}")
    # Satır sayısını yazdırmak, özellikle büyük dosyalarda işlemin ne kadar sürebileceği
    # hakkında fikir verir. Ayrıca olası dosya okuma hatalarını hemen fark etmemizi sağlar
    # (örneğin boş dosya okunursa 0 satır görürüz).

    # =============================================================
    # TEXT KONTROL
    # =============================================================
    # Modelin eğitildiği sütun adı ile tahmin yapılacak dosyadaki sütun adı aynı olmalı.
    # Bu kontrol, yanlış sütun adı girilmesi durumunda erken ve anlamlı bir hata verir.
    if text_col not in df.columns:
        raise ValueError(f"{text_col} kolonu bulunamadı!")

    # Null temizleme:
    # Eğitim sırasında NaN metinleri temizlemiştik (dropna). Tahmin sırasında da
    # aynı standardı korumalıyız. Burada NaN'leri boş string'e ("") çeviriyoruz.
    # Boş string, TF-IDF vektörleştiricide tüm değerleri 0 olan bir vektör üretir,
    # bu da modelin güvenli bir şekilde tahmin yapmasını sağlar.
    # Alternatif: NaN'li satırları tamamen silmek. Ancak bu, submission formatında
    # eksik satırlara yol açar ve genellikle istenmez. Doldurup tahmin yapmak daha güvenlidir.
    df = df.with_columns(
        pl.col(text_col)
        .fill_null("")
        .cast(pl.Utf8)  # Metin tipini garanti altına al. Model string bekler.
    )

    # Polars DataFrame'den sadece metin sütununu Python listesine çıkar.
    # .to_list() kullanıyoruz çünkü sklearn pipeline'ı doğrudan Polars Series
    # ile çalışmaz; pandas Series, numpy array veya Python listesi bekler.
    # Liste, en basit ve en hızlı dönüşümdür.
    texts = df[text_col].to_list()

    # =============================================================
    # MODEL YÜKLE
    # =============================================================
    print("Model yükleniyor...")
    # joblib.load() ile diske kaydedilmiş pipeline'ı geri yüklüyoruz.
    # Bu pipeline, içinde hem TF-IDF vektörleştiriciyi (dağarcık/vocabulary ile)
    # hem de eğitilmiş Lojistik Regresyon modelini barındırır.
    # Yükleme süresi, model dosyasının boyutuna bağlıdır (genelde birkaç saniye).
    # Production ortamında bu model bir kere yüklenip bellekte tutulur (örn. Flask/Django
    # uygulaması başlatılırken), her istekte tekrar yüklenmez.
    model = joblib.load(model_path)

    # =============================================================
    # TAHMİN
    # =============================================================
    print("Tahmin alınıyor...")

    # predict_proba(): Her metin için [organik_olasılığı, manipülatif_olasılığı] döndürür.
    # [:, 1] ile sadece ikinci kolonu (manipülatif olma olasılığı) alıyoruz.
    # Bu olasılık 0.0 ile 1.0 arasındadır. 0.0 = tamamen organik, 1.0 = tamamen manipülatif.
    # 
    # Neden predict() değil de predict_proba()?
    # - Yarışma formatı genelde olasılık skoru ister (ranking/leaderboard için).
    # - Olasılık skoru, iş birimlerine daha esnek bir kullanım sunar:
    #   "0.95 üstü → kesin manipülatif, 0.70-0.95 → incelemeye al, 0.50-0.70 → izle"
    #   gibi kademeli aksiyon planları yapılabilir.
    # - Sabit bir eşik değeri (0.5) ile ikili tahmin yapmak her zaman optimal değildir;
    #   precision-recall dengesi iş ihtiyacına göre sonradan ayarlanabilir.
    probs = model.predict_proba(texts)[:, 1]

    # Tahmin edilen olasılıkları yeni bir sütun olarak Polars DataFrame'e ekle.
    # pl.Series() ile numpy array'ini Polars serisine çeviriyoruz.
    df = df.with_columns(
        pl.Series("manipulation_probability", probs)
    )

    # =============================================================
    # OUTPUT FORMAT
    # =============================================================
    # İki farklı çıktı formatı:
    # 1) Yarışma/Submission formatı: id + probability
    #    Çoğu platform (Kaggle, vb.) bu formatı bekler.
    #    ID, sonuçların doğru gönderiyle eşleştirilmesi için zorunludur.
    # 2) Analiz formatı: text + probability
    #    ID yoksa veya son kullanıcı hangi metnin hangi skoru aldığını görmek
    #    istiyorsa bu format kullanılır. Manuel inceleme için idealdir.
    if id_col in df.columns:
        print("ID bulundu → id + probability output")
        # Sadece ID ve olasılık skorunu içeren minimal bir DataFrame.
        # Gereksiz sütunlar çıktıya yazılmaz, dosya boyutu küçük kalır.
        result = df.select([
            id_col,
            "manipulation_probability"
        ])

    else:
        print("ID yok → text + probability output")
        # Metinle birlikte çıktı vermek, tahminleri doğrulamak için kullanışlıdır.
        # "Bu metin neden 0.95 aldı?" sorusunun cevabı doğrudan dosyada görülebilir.
        result = df.select([
            text_col,
            "manipulation_probability"
        ])

    # =============================================================
    # KAYDET
    # =============================================================
    # Polars'ın write_csv() metodu, Pandas'tan çok daha hızlıdır.
    # Büyük dosyalarda (1M+ satır) fark hissedilir düzeydedir.
    result.write_csv(output_path)

    print("\nKaydedildi:", output_path)

    # Hızlı önizleme (preview):
    # Çıktının ilk 5 satırını yazdırarak, işlemin doğru tamamlandığını
    # görsel olarak doğrulamak için. Olasılıklar 0-1 arasında mı?
    # Sıralama mantıklı mı? gibi kontrolleri hemen yapabiliriz.
    print("\nÖrnek çıktı:")
    print(result.head(5))

    return result


# =============================================================
# ANA ÇALIŞTIRMA BLOĞU
# =============================================================
# Bu kısım, script doğrudan çalıştırıldığında (python inference.py) devreye girer.
# Bir modül olarak import edildiğinde (from inference import run_inference) çalışmaz.
# 
# Bu ayrım şu nedenle önemlidir:
# - Script olarak çalıştırırken varsayılan parametrelerle hızlıca test yapabiliriz.
# - Bir API/uygulama içinde kullanırken, run_inference() fonksiyonunu parametrelerle
#   çağırırız, bu alttaki çağrıyı yapmayız.
#
# NOT: Bu satırlar bir __name__ == "__main__" bloğu içinde OLMADIĞI için,
# bu dosya başka bir yerden import edilirse run_inference() FONKSİYONU ÇALIŞIR
# ve test.csv dosyası üzerinde inference yapar!
# 
# Bu genellikle İSTENMEYEN bir durumdur çünkü:
# - Import eden script, kendi dosya yoluyla çağırmak ister.
# - test.csv diye bir dosya import eden ortamda olmayabilir ve hata verir.
# 
# Ancak şu anki haliyle, bu script'in "hem modül hem script" olarak kullanılması
# amaçlanmıştır. Eğer sadece modül olarak kullanılacaksa, bu çağrının
# `if __name__ == "__main__":` bloğuna alınması gerekir.
# 
# Şu anki form, "Bu script'i çalıştırdığımda hemen inference yapsın, ayrıca
# başka yerden de import edip fonksiyonu çağırabileyim" esnekliğini sunar.
run_inference(
    input_path="test.csv",
    text_col="text",
    id_col="test_id",
    output_path="submission.csv"
)